<a href="https://colab.research.google.com/github/2410072/Python-Lesson/blob/main/RF_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# センサ由来データを用いた機械学習パイプライン
# Dataset : Breast Cancer Wisconsin Diagnostic (sklearn 同梱)
#           ※ 細胞核画像から計測された定量的特徴量データ
# Model   : Random Forest + SHAP による解釈性分析
# 環境    : Google Colab (ローカル保存なし)
# ============================================================

# --- SHAP のみ Colab に未導入のためインストール ---
!pip install -q shap

# --- ライブラリのインポート ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import shap

# 再現性確保のための乱数シード固定
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ============================================================
# 1. データセットの読み込み（メモリ上に直接ロード）
# ============================================================
data = load_breast_cancer(as_frame=True)
X = data.data       # (569, 30)
y = data.target     # 0: malignant, 1: benign
target_names = data.target_names

print(f"サンプル数: {X.shape[0]}, 特徴量数: {X.shape[1]}")
print(f"クラス: {dict(enumerate(target_names))}")

# ============================================================
# 2. データセット読み込み直後の可視化
# ============================================================
# データフレームを統合し、ラベル列を付与
df_full = X.copy()
df_full["target"] = y
df_full["diagnosis"] = df_full["target"].map({0: target_names[0], 1: target_names[1]})

# --- 2.1 先頭サンプルおよびデータ型の確認 ---
print("\n=== データフレーム先頭5件 ===")
print(df_full.head())
print("\n=== データ型と非欠損数 ===")
df_full.info()

# --- 2.2 欠損値の可視化（ヒートマップ） ---
plt.figure(figsize=(12, 4))
sns.heatmap(df_full.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing Value Heatmap (yellow = missing)")
plt.tight_layout()
plt.show()
print(f"欠損値の総数: {df_full.isnull().sum().sum()}")

# --- 2.3 クラス分布の可視化 ---
plt.figure(figsize=(5, 4))
sns.countplot(x="diagnosis", data=df_full, palette=["salmon", "steelblue"])
plt.title("Class Distribution")
plt.tight_layout()
plt.show()

# --- 2.4 全30特徴量のクラス別箱ひげ図 ---
# 標準化後にプロットすることで、スケールの異なる特徴量を同一軸上で比較可能にする
df_melt = df_full.drop(columns=["target"]).melt(id_vars="diagnosis",
                                                var_name="feature", value_name="value")
# 特徴量ごとに z-score 標準化
df_melt["value"] = df_melt.groupby("feature")["value"].transform(
    lambda v: (v - v.mean()) / v.std()
)

plt.figure(figsize=(16, 7))
sns.boxplot(x="feature", y="value", hue="diagnosis", data=df_melt,
            palette=["salmon", "steelblue"], fliersize=2)
plt.xticks(rotation=90)
plt.title("Standardized Feature Distributions by Class (All 30 Features)")
plt.tight_layout()
plt.show()

# ============================================================
# 3. 特徴量の絞り込み（解釈性確保のため "mean" 系特徴量に限定）
# ============================================================
mean_cols = [c for c in X.columns if c.startswith("mean ")]
X_sel = X[mean_cols].copy()
print(f"\n選択された特徴量 ({len(mean_cols)}個): {mean_cols}")

# --- 3.1 ペアプロットによる多変量分布の俯瞰 ---
# 全10特徴量だと描画が重いため、代表的な5特徴量に絞って可視化
pairplot_cols = ["mean radius", "mean texture", "mean perimeter",
                 "mean smoothness", "mean concavity"]
df_pair = X_sel[pairplot_cols].copy()
df_pair["diagnosis"] = df_full["diagnosis"]

sns.pairplot(df_pair, hue="diagnosis", palette=["salmon", "steelblue"],
             diag_kind="kde", plot_kws={"alpha": 0.6, "s": 20})
plt.suptitle("Pair Plot of Representative Features", y=1.02)
plt.show()

# ============================================================
# 4. 統計的分析（記述統計と相関分析）
# ============================================================
# --- 4.1 記述統計量 ---
print("\n=== 記述統計量 ===")
print(X_sel.describe().T)

# --- 4.2 相関分析（ヒートマップ） ---
corr_matrix = X_sel.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix of Selected Features")
plt.tight_layout()
plt.show()

# 強相関ペア(|r|>0.8)の抽出
strong_corr = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack().reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "corr"})
)
strong_corr = strong_corr[strong_corr["corr"].abs() > 0.8] \
    .sort_values("corr", key=abs, ascending=False)
print("\n=== 強相関ペア (|r| > 0.8) ===")
print(strong_corr.to_string(index=False))

# ============================================================
# 5. ランダムフォレストによる分類
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=target_names))

# 混同行列
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# ============================================================
# 6. ツリー構造の可視化（第1木 / 上位3段に限定）
# ============================================================
plt.figure(figsize=(20, 9))
plot_tree(
    rf_model.estimators_[0],
    max_depth=3,
    feature_names=X_sel.columns,
    class_names=target_names,
    filled=True, rounded=True, fontsize=10
)
plt.title("Visualization of a Single Decision Tree (Top 3 Levels)")
plt.show()

# ============================================================
# 7. SHAPによる特徴量重要度分析
# ============================================================
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# SHAP値の形状を統一的に扱うための補助処理
if isinstance(shap_values, list):
    sv_class1 = shap_values[1]                  # benign クラス
else:
    sv_class1 = shap_values[:, :, 1]

# --- 7.1 特徴量重要度（Bar Plot） ---
shap.summary_plot(sv_class1, X_test, plot_type="bar", show=True)

# --- 7.2 SHAP値の分布（Beeswarm Plot） ---
shap.summary_plot(sv_class1, X_test, show=True)

print("\n--- パイプライン実行完了 ---")
